In [ ]:
import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt
# import matplotlib.ticker as ticker
# import seaborn as sns

import json

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, callbacks

# import gc

from preprocessor import TextEncoder
import pickle

In [2]:
# Load pre-cleaned data
cars_clean=pd.read_csv('cars_clean.csv')

In [3]:
# ==========================================
# 1. SETUP & DATA PREP
# ==========================================


print("--- Preparing Data ---")

# 1. Shuffle & Split
shuffled_df = cars_clean.sample(frac=1, random_state=42).reset_index(drop=True)

train_size = int(0.8 * len(shuffled_df))
val_size = int(0.1 * len(shuffled_df))

train_df = shuffled_df.iloc[:train_size]
val_df = shuffled_df.iloc[train_size:train_size+val_size]
test_df = shuffled_df.iloc[train_size+val_size:]

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

--- Preparing Data ---
Train: 610848 | Val: 76356 | Test: 76356


In [13]:
# ==========================================
# 2. VOCABULARY & ENCODING  (Depends on the training data)
# ==========================================



print("--- Building Vocabularies ---")

# Set relevant features
cat_features = ['make', 'model', 'fuel_type']
num_features = ['miles', 'year']

# Keep only relevant columns for modeling
train_df = train_df[cat_features + num_features + ['car_price']]
val_df = val_df[cat_features + num_features + ['car_price']]
test_df = test_df[cat_features + num_features + ['car_price']]

# Make an encoder
encoder = TextEncoder()

# Fit the encoder on the training data (builds vocabularies for the categorical features set above)
encoder.fit(cars_clean, cat_features)

# Save/update json of strings to control app dropdowns
make_model = { make : cars_clean.loc[cars_clean['make'] ==  make, 'model'].unique().tolist() for make in cars_clean['make'].unique() }
fuel_types = cars_clean['fuel_type'].unique().tolist()
with open('ui_assets.json', 'w') as f:
    json.dump({'make_model': make_model, 'fuel_types': fuel_types}, f, indent=4)

# Pickle the encoder function for later predictions
with open('encoder.pkl', 'wb') as f:
    pickle.dump(encoder, f)


# Encode categorical features and combine with numerical features for model input
train_inputs = encoder.transform_df(train_df) | {'num_in': train_df[num_features].values.astype('float32')}
val_inputs = encoder.transform_df(val_df) | {'num_in': val_df[num_features].values.astype('float32')}
test_inputs = encoder.transform_df(test_df) | {'num_in': test_df[num_features].values.astype('float32')}


# We train the model to predict log(price), not raw price.
train_targets_log = np.log(train_df['car_price'].values.astype('float32'))
val_targets_log = np.log(val_df['car_price'].values.astype('float32'))
test_targets_raw = test_df['car_price'].values.astype('float32') # Keep raw for final eval



# Clean up RAM
# del shuffled_df, cars_clean
# gc.collect()



--- Building Vocabularies ---


['make', 'model', 'fuel_type']

In [18]:

# ==========================================
# 3. MODEL ARCHITECTURE
# ==========================================
print("--- Building Model ---")

# --- Branch A: Make ---
in_make = layers.Input(shape=(1,), name='make')
emb_make = layers.Flatten()(layers.Embedding(len(encoder.vocabs['make'])+1, 10)(in_make))

# --- Branch B: Model ---
in_model = layers.Input(shape=(1,), name='model')
emb_model = layers.Flatten()(layers.Embedding(len(encoder.vocabs['model'])+1, 15)(in_model))

# --- Branch C: Variant --- 
# in_variant = layers.Input(shape=(1,), name='variant')
# emb_variant = layers.Flatten()(layers.Embedding(len(encoder.vocabs['variant'])+1, 10)(in_variant))

# --- Branch D: Fuel Type --- 
in_fuel = layers.Input(shape=(1,), name='fuel_type')
# Few categories (Petrol, Diesel, EV...), so small vector (3-5) is fine
emb_fuel = layers.Flatten()(layers.Embedding(len(encoder.vocabs['fuel_type'])+1, 5)(in_fuel))

# --- Branch E: Numericals ---
in_num = layers.Input(shape=(2,), name='num_in')
normalizer = layers.Normalization(axis=-1)
normalizer.adapt(train_inputs['num_in'])
norm_num = normalizer(in_num)


# --- Merge All 5 Branches ---
merged = layers.Concatenate()([emb_make, emb_model, emb_fuel, norm_num])

# --- Dense Head ---
x = layers.Dense(128, activation='relu')(merged)
x = layers.Dropout(0.2)(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.1)(x)


output = layers.Dense(1, activation='linear', name='log_price_out')(x)

model = Model(inputs=[in_make, in_model, in_fuel, in_num], outputs=output)

# Using Mean Squared Error for Log-Space regression is standard
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mean_absolute_error'])

--- Building Model ---


In [19]:




# ==========================================
# 4. TRAINING
# ==========================================
print("--- Starting Training ---")
history = model.fit(
    x=train_inputs,
    y=train_targets_log, # Training on Log Targets!
    validation_data=(val_inputs, val_targets_log),
    epochs=50, # Set high, let EarlyStopping handle it
    batch_size=512,
    callbacks=[
        callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)
    ],
    verbose=1
)

# ==========================================
# 5. EVALUATION (undoing the log)
# ==========================================
print("\n--- Final Evaluation ---")

# 1. Predict (Output is in Log Space)
log_preds = model.predict(test_inputs, batch_size=512)

# 2. Inverse Transform (Log -> GBP)
# expm1 reverses log1p
gbp_preds = np.exp(log_preds).flatten()

# 3. Calculate Error in Real Money
mae = np.mean(np.abs(test_targets_raw - gbp_preds))
mape = np.mean(np.abs((test_targets_raw - gbp_preds) / test_targets_raw)) * 100

print(f"Final MAE: £{mae:,.2f}")
print(f"Final MAPE: {mape:.2f}% (Average percentage error)")

# 4. Show a few examples
results = pd.DataFrame({'Actual': test_targets_raw, 'Predicted': gbp_preds})
results['Error'] = results['Actual'] - results['Predicted']
print("\n--- Sample Predictions ---")
print(results.head(10))

--- Starting Training ---
Epoch 1/50
1194/1194 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 2.8052 - mean_absolute_error: 0.9339 - val_loss: 0.0642 - val_mean_absolute_error: 0.1843 - learning_rate: 0.0010
Epoch 2/50
1194/1194 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 0.5589 - mean_absolute_error: 0.5927 - val_loss: 0.0506 - val_mean_absolute_error: 0.1600 - learning_rate: 0.0010
Epoch 3/50
1194/1194 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 0.4918 - mean_absolute_error: 0.5559 - val_loss: 0.0493 - val_mean_absolute_error: 0.1606 - learning_rate: 0.0010
Epoch 4/50
1194/1194 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 0.4029 - mean_absolute_error: 0.5025 - val_loss: 0.0412 - val_mean_absolute_error: 0.1450 - learning_rate: 0.0010
Epoch 5/50
1194/1194 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 0.3307 - mean_absolute_error: 0.4544 - val_loss: 0.0417 - val_mean_absolute_error: 0.1429 - learning_rate: 0.0010
Epoch 6/50
1194/1194 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 0.2929 - mean_absolute_error: 0.

In [ ]:
model.save('savedModels/noVariant.keras')

In [51]:
single_input = encoder.transform_dict({'make' : 'Toyota', 'model' : 'Yaris', 'fuel_type': 'petrol'})
model.predict(single_input | {'num_in': np.array([[2007, 2495]], dtype='float32')})

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step


array([[71.24577]], dtype=float32)

In [58]:
input = encoder.transform_df(pd.DataFrame({'make': ['Toyota'], 'model': ['Yaris'], 'fuel_type': ['petrol'], 'miles': [2495], 'year': [2007]})) | {'num_in': np.array([[2495, 2007]], dtype='float32')}
np.exp(model.predict(input))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step


array([[5997.286]], dtype=float32)

In [41]:
cars_clean.loc[(cars_clean.make == 'Toyota') & (cars_clean.model == 'Yaris')][['fuel_type','miles','car_price', 'year']]

,fuel_type,miles,car_price,year
684497,petrol,88000.0,5500.0,2015
684498,petrol,61900.0,6375.0,2015
684499,petrol,43000.0,6500.0,2015
684500,petrol,46000.0,6995.0,2015
684501,petrol,52000.0,6995.0,2014
...,...,...,...,...
691778,petrol,127000.0,2195.0,2007
691779,petrol,87000.0,2275.0,2007
691780,petrol,62314.0,2395.0,2007
691781,petrol,112000.0,2399.0,2007


In [52]:
inputs = encoder.transform_df(cars_clean.loc[(cars_clean.car_price == 8495)][['make', 'model', 'fuel_type','miles', 'year','car_price']]) | {'num_in': cars_clean.loc[(cars_clean.car_price == 8495)][['miles', 'year']].values.astype('float32')}
(model.predict(inputs))

82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


array([[9.07023 ],
       [9.168235],
       [9.006684],
       ...,
       [9.078988],
       [9.019501],
       [9.019501]], shape=(2624, 1), dtype=float32)

In [61]:
encoder.transform_dict({'make' : 'Toyota', 'model' : 'Yaris', 'fuel_type': 'petrol'})


{'make': array([[122]]), 'model': array([[1265]]), 'fuel_type': array([[7]])}

In [10]:
inputs = encoder.transform_df(pd.DataFrame({'make': ['Toyota', 'Ford', 'Ford'], 'model': ['Yaris', 'Fiesta', 'Focus'], 'fuel_type': ['petrol', 'petrol', 'diesel']}))| {'num_in': np.array([[2495, 2015 ], [20000, 2020], [12500, 2009]], dtype='float32')}

In [8]:
model = tf.keras.models.load_model('models/noVariant.keras')

In [11]:
np.exp(model.predict(inputs))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 308ms/step


array([[10340.338],
       [15016.19 ],
       [ 7270.273]], dtype=float32)